# Re-run: validity (Paper II, RQ3)

This notebook recomputes the headline numbers of RQ3 from the committed outputs in the code package of
release `learning-graph-budget-2026`. It needs Python 3.9+ with numpy and the unzipped release (run it
from the release folder or its `notebooks/` folder, or set `RELEASE_DIR`).

**What it recomputes, and from what.** Every estimate here was fitted on a corpus that may not be
redistributed (ASSISTments 2012–13, EDM Cup, Junyi, the ASSISTments experiments, DataShop dataset 1190,
the length-measurement trial's archive); `reproduce.py` refits them for a reader who holds them
(steps `rq3-*`). The release keeps each fit's parts, and the notebook recomputes the printed results
from them:

- Table 8: each excess as the residualized contrast minus its deranged arm, the noise as twice the
  standard deviation of the excess in the synthetic null, the verdict from the pre-registered rule,
  and the number of pairs that would resolve the bar from the fitted noise constant;
- Junyi: the raters' panel reliability stepped up with Spearman–Brown, and the correlation of rating
  with the learner order effect partialled on taxonomy distance from its three correlations;
- Table 9: the four-order experiment's T1 − T3 contrast from the arm means, the covariate's variance
  ratio from the two standard errors, and the fraction swap's contrast from the two arms' accuracies;
- the trial sizes: the per-arm sizes from the normal approximation 2(z₀.₉₇₅ + z₀.₈₀)²/d², with the
  absolute drop of 0.10 converted to standard deviations at accuracies 0.5 and 0.8, and the
  adjustments for the covariate and for randomizing classes.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os
from pathlib import Path

RELEASE_ID = "learning-graph-budget-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Code package:", shown(code_dir()))

Code package: code


In [2]:
import json, math
import numpy as np

sg = lambda x, d=4: f"{x:+.{d}f}".replace("-", "−")
assist = load_code_json("studies/measurement/assist2012-direction/out/direction.json")
edm = load_code_json("studies/measurement/edmcup-direction/out/direction.json")

def verdict(s, bar):
    """The pre-registered rule: recovered when the excess clears the bar and its interval excludes zero;
    a null whose bar lies below the instrument's noise is underpowered, not a negative."""
    if s.get("excess") is None:
        return "UNANSWERABLE"
    if s["excess"] > bar and s["ci_lo"] > 0:
        return "RECOVERED"
    return "UNDERPOWERED AT THE BAR" if 2 * s["floor_null_sd"] > bar else "NOT RECOVERED"

PRINTED_T8 = [  # pairs, excess [interval], noise, planted gate recovered, verdict
    (assist, 0, (1426, "+0.0117 [+0.0054, +0.0181]", "0.0033", "+0.1502", "NOT RECOVERED")),
    (assist, 1, (472, "−0.0022 [−0.0099, +0.0094]", "0.0069", "+0.1495", "NOT RECOVERED")),
    (edm, 0, (5, None, None, None, "UNANSWERABLE")),
    (edm, 1, (54, "−0.0291 [−0.0781, +0.0164]", "0.0305", "+0.1558", "UNDERPOWERED AT THE BAR")),
    (edm, 2, (71, "+0.0103 [−0.0334, +0.0491]", "0.0251", "+0.1487", "UNDERPOWERED AT THE BAR"))]
for src, i, want in PRINTED_T8:
    s = src["structures"][i]
    if s.get("excess") is None:
        got = (s["pairs"], None, None, None, verdict(s, src["bar"]))
    else:
        excess = s["delta_residualised"] - s["deranged_residualised"]      # recomputed, then checked
        noise = 2 * s["floor_null_sd"]
        assert abs(excess - s["excess"]) < 1e-12 and abs(noise - s["noise_2sd"]) < 1e-12
        got = (s["pairs"], f"{sg(excess)} [{sg(s['ci_lo'])}, {sg(s['ci_hi'])}]", f"{noise:.4f}", sg(s["floor_planted"]),
               verdict(s, src["bar"]))
    print(f"{s['structure']:40s} {got}")
    assert got == want and got[-1] == s["verdict"], (s["structure"], got)
assert assist["bar"] == edm["bar"] == 0.02
for name, src, want in (("ASSISTments", assist, (45, None)), ("EDM Cup", edm, (126, 504))):
    c = src["pricing"]["noise_constant"]
    for_bar = math.ceil((c / 0.02) ** 2)               # noise c / sqrt(pairs) reaches the bar of 0.02
    half = math.ceil((c / 0.01) ** 2)
    print(f"{name}: noise {c:.3f}/sqrt(pairs): {for_bar} pairs resolve the bar, {half} half of it; {src['pricing']['pairs_in_hand']:,} in hand")
    assert for_bar == src["pricing"]["pairs_for_bar"] == want[0]
    assert want[1] is None or half == src["pricing"]["pairs_for_half_bar"] == want[1]
assert (f"{assist['pricing']['noise_constant']:.3f}", f"{edm['pricing']['noise_constant']:.3f}") == ("0.134", "0.224")
strata = assist["structures"][0]["strata"]
got = [(v["pairs"], sg(v["delta"] - v["deranged"])) for v in strata.values()]
print("by grade gap:", got)
assert got == [(581, "+0.0085"), (470, "+0.0160"), (216, "+0.0293"), (159, "+0.0266")]
print("Table 8, the verdicts and the grade-gap strata match the paper.")

S1  CCSS GRADE                           (1426, '+0.0117 [+0.0054, +0.0181]', '0.0033', '+0.1502', 'NOT RECOVERED')
S1E CCSS GRADE, EXACT NAMES ONLY         (472, '−0.0022 [−0.0099, +0.0094]', '0.0069', '+0.1495', 'NOT RECOVERED')
S1  CCSS GRADE                           (5, None, None, None, 'UNANSWERABLE')
S2  CURRICULUM MODULE ORDER              (54, '−0.0291 [−0.0781, +0.0164]', '0.0305', '+0.1558', 'UNDERPOWERED AT THE BAR')
S3  CURRICULUM LESSON ORDER              (71, '+0.0103 [−0.0334, +0.0491]', '0.0251', '+0.1487', 'UNDERPOWERED AT THE BAR')
ASSISTments: noise 0.134/sqrt(pairs): 45 pairs resolve the bar, 180 half of it; 1,754 in hand
EDM Cup: noise 0.224/sqrt(pairs): 126 pairs resolve the bar, 504 half of it; 128 in hand
by grade gap: [(581, '+0.0085'), (470, '+0.0160'), (216, '+0.0293'), (159, '+0.0266')]
Table 8, the verdicts and the grade-gap strata match the paper.


## Junyi: the raters, the learners and taxonomy distance

In [3]:
jx = load_code_json("studies/identifiability/junyi/junyi-expert-vs-data.json")
r = jx["rater_split_half"]
print(f"raters: split-half {r:.3f}, stepped up {2 * r / (1 + r):.3f}")
assert (f"{r:.3f}", f"{2 * r / (1 + r):.3f}") == ("0.783", "0.879")
rho, ci = jx["spearman"], jx["ci"]
print(f"rating vs learner order effect over {jx['n_pairs_learner']} pairs: {sg(rho, 3)} [{sg(ci[0], 3)}, {sg(ci[1], 3)}]; "
      f"swapped {sg(jx['reversed_arrow']['rated_vs_learners_rho']['reversed'], 3)}")
assert (jx["n_pairs_learner"], sg(rho, 3), sg(ci[0], 3), sg(ci[1], 3)) == (47, "+0.299", "−0.017", "+0.571")
assert ci[0] < 0 < ci[1] and sg(jx["reversed_arrow"]["rated_vs_learners_rho"]["reversed"], 3) == "−0.059"

tc = load_code_json("studies/identifiability/junyi-taxonomy-control/out/taxonomy-control.json")
t = tc["treatment"]
r_tg, r_tD, r_gD = t["rho_t"], t["rating_vs_dist"], t["gap_vs_dist"]
partial = (r_tg - r_tD * r_gD) / math.sqrt((1 - r_tD ** 2) * (1 - r_gD ** 2))
print(f"with ties averaged {sg(r_tg, 3)}; rating vs distance {sg(r_tD, 3)}, learner effect vs distance {sg(r_gD, 3)}; "
      f"given distance {sg(partial, 3)}")
assert abs(partial - tc["partial"]["rho_p"]) < 1e-9
assert (sg(r_tg, 3), sg(r_tD, 3), sg(r_gD, 3), sg(partial, 3)) == ("+0.297", "−0.635", "−0.322", "+0.127")
c2 = tc["C2"]
assert (sg(c2["rho_c"], 3), sg(c2["rho_t"] - c2["rho_c"], 3)) == ("+0.231", "+0.066")
allr = tc["rating_vs_dist_all_rows"]
means = [allr["mean_rating_by_dist"][k] for k in ("0", "1", "2")]
assert (sg(allr["spearman"], 3), [f"{m:.2f}" for m in means]) == ("−0.740", ["5.96", "3.71", "2.32"])
print("The Junyi numbers match: +0.299, partialled +0.127, distance −0.740, mean ratings 5.96, 3.71, 2.32.")

raters: split-half 0.783, stepped up 0.879
rating vs learner order effect over 47 pairs: +0.299 [−0.017, +0.571]; swapped −0.059
with ties averaged +0.297; rating vs distance −0.635, learner effect vs distance −0.322; given distance +0.127
The Junyi numbers match: +0.299, partialled +0.127, distance −0.740, mean ratings 5.96, 3.71, 2.32.


## Randomized tests of order (Table 9)

In [4]:
lo = load_code_json("studies/outcomes/lt-trials/out/length_order.json")
m = lo["authors_model"]["vs_REV"]["LT"]
print(f"length trial: trajectory minus reverse {sg(m['coef'], 3)} logits [{sg(m['lo'], 3)}, {sg(m['hi'], 3)}]; "
      f"class bootstrap {lo['lt_vs_rev']['cluster_bootstrap_95']}")
assert (lo["n"], sg(m["coef"], 3), sg(m["lo"], 3), sg(m["hi"], 3)) == (187, "+0.317", "+0.067", "+0.566")
assert lo["lt_vs_rev"]["cluster_bootstrap_95"][0] < 0                  # the class-resampled interval includes zero

os_ = load_code_json("studies/identifiability/core-vs-culture/out/order-swap-result.json")["problems_to_mastery"]
d13 = os_["means"][0] - os_["means"][2]                               # Treatment 1 minus Treatment 3, from the arm means
assert abs(d13 - os_["pairs"]["T1-T3"]["diff"]) < 1e-12
print(f"four orders: arm means {[f'{x:.2f}' for x in os_['means']]}; T1 - T3 {d13:+.2f} "
      f"[90%: {os_['pairs']['T1-T3']['lo90']:+.2f}, {os_['pairs']['T1-T3']['hi90']:+.2f}]; omnibus p {os_['omnibus_p']:.4f}")
assert [f"{x:.2f}" for x in os_["means"]] == ["6.51", "7.16", "8.00", "6.93"]
assert (sg(d13, 2), sg(os_["pairs"]["T1-T3"]["lo90"], 2), sg(os_["pairs"]["T1-T3"]["hi90"], 2)) == ("−1.49", "−2.24", "−0.77")

rem = load_code_json("studies/outcomes/order-swap-remnant/out/remnant-result.json")["problems_to_mastery"]["pairs"]["T1-T3"]
ratio = (rem["adjusted"]["se"] / rem["unadjusted"]["se"]) ** 2
assert abs(ratio - rem["variance_ratio"]) < 1e-12
print(f"remnant-adjusted: {rem['adjusted']['diff']:+.2f} [90%: {rem['adjusted']['lo90']:+.2f}, {rem['adjusted']['hi90']:+.2f}]; "
      f"variance ratio {ratio:.3f}, i.e. {100 * (1 / ratio - 1):.0f}% more students")
assert (sg(rem["adjusted"]["diff"], 2), sg(rem["adjusted"]["lo90"], 2), sg(rem["adjusted"]["hi90"], 2)) == ("−1.47", "−2.16", "−0.81")
assert (f"{ratio:.3f}", f"{100 * (1 / ratio - 1):.0f}") == ("0.828", "21")

pa = load_code_json("studies/outcomes/order-trials-reanalysis/out/patel_answers.json")["per_test"]["DPosttest"]["trained_primary"]
d = pa["ma"] - pa["am"]
assert abs(d - pa["ma_minus_am"]) < 1e-12
print(f"fraction swap: multiplication first {pa['ma']:.3f} (n {pa['n_ma']}), addition first {pa['am']:.3f} (n {pa['n_am']}): "
      f"{d:+.3f} [{pa['ci95_ma_minus_am'][0]:+.3f}, {pa['ci95_ma_minus_am'][1]:+.3f}], p {pa['p']:.3f}; {pa['verdict']}")
assert (f"{pa['ma']:.3f}", f"{pa['am']:.3f}", sg(d, 3), pa["n_am"] + pa["n_ma"]) == ("0.772", "0.683", "+0.089", 55)
assert (sg(pa["ci95_ma_minus_am"][0], 3), sg(pa["ci95_ma_minus_am"][1], 3), f"{pa['p']:.3f}") == ("−0.025", "+0.203", "0.144")
print("Table 9's estimates match the paper.")

length trial: trajectory minus reverse +0.317 logits [+0.067, +0.566]; class bootstrap [-0.008564042644489992, 0.5078841236793745]
four orders: arm means ['6.51', '7.16', '8.00', '6.93']; T1 - T3 -1.49 [90%: -2.24, -0.77]; omnibus p 0.0035
remnant-adjusted: -1.47 [90%: -2.16, -0.81]; variance ratio 0.828, i.e. 21% more students
fraction swap: multiplication first 0.772 (n 29), addition first 0.683 (n 26): +0.089 [-0.025, +0.203], p 0.144; UNINFORMATIVE (BELOW FLOOR)
Table 9's estimates match the paper.


## How many learners a trial of one edge needs

Per arm, n = 2Z²/d² with Z = z₀.₉₇₅ + z₀.₈₀. An absolute drop of 0.10 is d = 0.10/σ standard
deviations, with σ² the mean of the two arms' binomial variances at accuracies p and p − 0.10. The
covariate multiplies the per-learner variance by its variance ratio; randomizing classes of m learners
with intraclass correlation ρ multiplies it by ρ + (1 − ρ)/m per learner, so that n becomes classes.

In [5]:
pw = load_code_json("studies/outcomes/order-swap-remnant/out/power.json")
ip = load_code_json("studies/outcomes/lt-trials/out/icc-power.json")
from statistics import NormalDist
Z = NormalDist().inv_cdf(0.975) + NormalDist().inv_cdf(0.80)
assert abs(Z - pw["Z"]) < 1e-9
d_at = lambda p: 0.10 / math.sqrt((p * (1 - p) + (p - 0.1) * (1.1 - p)) / 2)
eta = rem["variance_ratio"]
rho_cls = pw["icc"]["problems_to_mastery_class"]["icc"]
rho_lt = ip["rho_used"]
def plan(d):
    base = 2 * Z ** 2 / d ** 2
    cls = lambda rho, m: math.ceil(base * (rho + (1 - rho) / m))
    return (math.ceil(base), math.ceil(base * eta), cls(rho_cls, 13), cls(rho_cls, 20), cls(rho_lt["length_trial_icc_point"], 13))
got = {"abs 0.10 at accuracy 0.5": plan(d_at(0.5)), "abs 0.10 at accuracy 0.8": plan(d_at(0.8)), "0.10 SD": plan(0.10)}
for k, v in got.items():
    print(f"{k:26s} learners {v[0]:5,d}, with the covariate {v[1]:5,d}; classes of 13: {v[2]}, of 20: {v[3]}; "
          f"at the length trial's ICC: {v[4]}")
print(f"d at 0.5 and 0.8: {d_at(0.5):.2f} and {d_at(0.8):.2f} SD")
assert (f"{d_at(0.5):.2f}", f"{d_at(0.8):.2f}") == ("0.20", "0.23")
assert got == {"abs 0.10 at accuracy 0.5": (385, 319, 64, 55, 170), "abs 0.10 at accuracy 0.8": (291, 241, 49, 42, 129),
               "0.10 SD": (1570, 1301, 260, 222, 693)}
pa_abs = load_code_json("studies/outcomes/order-swap-remnant/out/planning-abs.json")
assert pa_abs["abs_0.10"]["at_accuracy_0.5"]["learner_n_per_arm"] == 385 and pa_abs["sd_0.10"]["class_m13_icc_length_trial"] == 693
lt_range = [math.ceil(2 * Z ** 2 / 0.01 * (r + (1 - r) / 13)) for r in (rho_lt["length_trial_icc_lo90"], rho_lt["length_trial_icc_hi90"])]
assert lt_range == [448, 965]
print(f"class ICC {rho_cls:.3f} (four-order experiment), {rho_lt['length_trial_icc_point']:.3f} (length trial); "
      f"across the length trial's 90% interval, {lt_range[0]} to {lt_range[1]} classes of 13 per arm at 0.10 SD")
assert (f"{rho_cls:.3f}", f"{rho_lt['length_trial_icc_point']:.3f}") == ("0.096", "0.395")
print("The trial sizes match the paper: 291 to 385 learners per arm for an absolute drop of 0.10, 1,570 for 0.10 SD.")

abs 0.10 at accuracy 0.5   learners   385, with the covariate   319; classes of 13: 64, of 20: 55; at the length trial's ICC: 170
abs 0.10 at accuracy 0.8   learners   291, with the covariate   241; classes of 13: 49, of 20: 42; at the length trial's ICC: 129
0.10 SD                    learners 1,570, with the covariate 1,301; classes of 13: 260, of 20: 222; at the length trial's ICC: 693
d at 0.5 and 0.8: 0.20 and 0.23 SD
class ICC 0.096 (four-order experiment), 0.395 (length trial); across the length trial's 90% interval, 448 to 965 classes of 13 per arm at 0.10 SD
The trial sizes match the paper: 291 to 385 learners per arm for an absolute drop of 0.10, 1,570 for 0.10 SD.
